In [1]:
import os
import sys
import pandas as pd
from tqdm.notebook import tqdm

from utilities import get_date, walk_through_files, get_subzone_name_from_dir

pd.set_option('display.max_rows', 500)

In [2]:
RECORDS_PATH =  os.path.join('..', 'raw-files', 'sg_subzones_devices')
SAVEPATH = os.path.join('intermediate-files', 'devicecount-subzone-day.csv')

In [3]:
ix = 0
dict_placeholder = {}


for filepath in tqdm(walk_through_files(RECORDS_PATH, 'csv.gz', fullpath=True)):
    
    # parse file name to get date and loc 
    try:
        subzone = get_subzone_name_from_dir(filepath)
        date = get_date(filepath)
    except ValueError:
        print(filename)
    
    # read into df
    _df = pd.read_csv(filepath, compression='gzip', 
                      names=['id', 'os'], 
                      dtype={'id': 'object', 'os': 'int8'})
    
    android_count = len(_df[_df['os']==0])
    ios_count = len(_df[_df['os']==1])
    
    # init list and add to dict
    dict_placeholder[ix] = [subzone, date, android_count, ios_count]
    
    ix += 1

0it [00:00, ?it/s]

In [4]:
df = (pd.DataFrame.from_dict(dict_placeholder, orient='index',
                             columns=['subzone', 'date', 'android_count', 'ios_count'])
      .astype({'subzone': 'object', 
               'date': 'datetime64',
               'android_count': 'int', 
               'ios_count': 'int'})
      .sort_values(['date', 'subzone'], ignore_index=True)
     )

In [5]:
df.head(10)

,subzone,date,android_count,ios_count
0,admiralty,2020-01-01,63,10
1,airport road,2020-01-01,54,3
2,alexandra hill,2020-01-01,1080,203
3,alexandra north,2020-01-01,362,74
4,aljunied,2020-01-01,5687,1033
5,anak bukit,2020-01-01,1296,310
6,anchorvale,2020-01-01,1894,380
7,ang mo kio town centre,2020-01-01,1200,176
8,anson,2020-01-01,265,77
9,balestier,2020-01-01,3543,1289


In [6]:
df[df['android_count']<df['ios_count']]

,subzone,date,android_count,ios_count
21,boat quay,2020-01-01,987,1060
195,pulau punggol barat,2020-01-01,0,1
1217,boat quay,2020-01-05,811,1009
1516,boat quay,2020-01-06,988,1204
1815,boat quay,2020-01-07,1160,1229
2414,boat quay,2020-01-09,1225,1766
2588,pulau punggol barat,2020-01-09,0,1
2714,boat quay,2020-01-10,1282,1321
3013,boat quay,2020-01-11,1108,1503
3312,boat quay,2020-01-12,878,1180


In [7]:
df.groupby('subzone')['android_count'].sum().reset_index().sort_values('android_count').head(15)

,subzone,android_count
196,pulau punggol timor,1
195,pulau punggol barat,8
268,tuas promenade,256
270,tuas view extension,838
269,tuas view,1326
85,greenwood park,1740
231,simpang north,1872
60,coney island,2774
0,admiralty,3877
257,tengeh,4645


In [8]:
df.groupby('subzone')['android_count'].sum().reset_index().sort_values('android_count').tail(15)

,subzone,android_count
13,bedok north,312353
9,balestier,335906
300,yunnan,336417
4,aljunied,349524
276,upper thomson,353564
136,lorong ah soo,463472
244,tampines east,472077
224,serangoon garden,475476
126,kovan,574231
39,central water catchment,809609


In [9]:
df.to_csv(SAVEPATH, index=False)